# Optimizing Model Parameters

## Prerequisite Code

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

## Hyperparameters

In [2]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

## Optimization Loop

### Loss Function

In [3]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

### Optimizer

In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Full Implementation

In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with troch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.166303  [   64/60000]
loss: 2.152652  [ 6464/60000]
loss: 2.091683  [12864/60000]
loss: 2.118959  [19264/60000]
loss: 2.057978  [25664/60000]
loss: 2.001309  [32064/60000]
loss: 2.029562  [38464/60000]
loss: 1.951229  [44864/60000]
loss: 1.957726  [51264/60000]
loss: 1.887650  [57664/60000]
Test Error: 
 Accuracy: 57.0%, Avg loss: 1.882608 

Epoch 2
-------------------------------
loss: 1.911599  [   64/60000]
loss: 1.879709  [ 6464/60000]
loss: 1.759026  [12864/60000]
loss: 1.813776  [19264/60000]
loss: 1.693555  [25664/60000]
loss: 1.649741  [32064/60000]
loss: 1.671402  [38464/60000]
loss: 1.581144  [44864/60000]
loss: 1.601105  [51264/60000]
loss: 1.499648  [57664/60000]
Test Error: 
 Accuracy: 62.3%, Avg loss: 1.517457 

Epoch 3
-------------------------------
loss: 1.579922  [   64/60000]
loss: 1.545995  [ 6464/60000]
loss: 1.394791  [12864/60000]
loss: 1.475801  [19264/60000]
loss: 1.351792  [25664/60000]
loss: 1.351811  [32064/600